# HW8 Bonus Questions: Decision Trees

This notebook contains fresh, Colab-ready code for the two bonus questions.

- **BQ1:** Structural instability of decision trees using the **Ames Housing** dataset.
- **BQ2:** Trees as rule-based systems using the **Breast Cancer Wisconsin** dataset.


## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.tree import export_text, plot_tree
from sklearn.metrics import accuracy_score
from sklearn.metrics.cluster import adjusted_rand_score

# BQ1: Structural Instability of Decision Trees

The goal is to compare two decision trees trained on slightly different samples. Instead of only comparing predictions, we also compare the **partitions** created by the trees.

Two trees are structurally close if they group observations into similar leaves. We measure this using the **Adjusted Rand Index (ARI)** between leaf assignments on the same test set.

## Load Ames Housing Dataset

In [ ]:
# Robust Ames Housing loader.
# Primary option: OpenML Ames/Kaggle housing data.
# Fallback option: CSV mirror if OpenML is temporarily unavailable.

from sklearn.datasets import fetch_openml

try:
    ames_openml = fetch_openml(name="house_prices", as_frame=True, parser="auto")
    ames = ames_openml.frame.copy()
    print("Loaded Ames Housing from OpenML")
except Exception as e:
    print("OpenML failed, trying CSV fallback:", e)
    ames_url = "https://huggingface.co/datasets/Causal-Copilot/Causal-Copilot-Dataset/resolve/main/realworld_data/AmesHousing.csv"
    ames = pd.read_csv(ames_url)
    print("Loaded Ames Housing from CSV fallback")

print(ames.shape)
ames.head()

## Prepare Data

In [ ]:
target = "SalePrice"

X_ames = ames.drop(columns=[target])
y_ames = ames[target]

# Remove ID-like columns if present.
for col in ["Order", "PID", "Id"]:
    if col in X_ames.columns:
        X_ames = X_ames.drop(columns=[col])

# Convert categorical variables to dummy variables.
X_ames = pd.get_dummies(X_ames, drop_first=True)

# Fill missing numeric values with the median.
X_ames = X_ames.fillna(X_ames.median(numeric_only=True))

X_train, X_test, y_train, y_test = train_test_split(
    X_ames,
    y_ames,
    test_size=0.25,
    random_state=42
)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)

## Train Two Slightly Different Trees

In [ ]:
# Tree 1: trained on the full training data.
tree1 = DecisionTreeRegressor(random_state=1)
tree1.fit(X_train, y_train)

# Tree 2: trained on a slightly modified training set.
# Here we remove 10% of the training observations.
X_train_2 = X_train.sample(frac=0.90, random_state=2)
y_train_2 = y_train.loc[X_train_2.index]

tree2 = DecisionTreeRegressor(random_state=1)
tree2.fit(X_train_2, y_train_2)

pred1 = tree1.predict(X_test)
pred2 = tree2.predict(X_test)

prediction_difference = np.mean(np.abs(pred1 - pred2))

print("BQ1 Results")
print("-----------")
print("Average absolute prediction difference:", prediction_difference)
print("Tree 1 depth:", tree1.get_depth())
print("Tree 2 depth:", tree2.get_depth())
print("Tree 1 number of nodes:", tree1.tree_.node_count)
print("Tree 2 number of nodes:", tree2.tree_.node_count)

## Structural Comparison Using Leaf Assignments

Each test observation is assigned to a leaf node. If two trees produce similar leaf groupings, then their partitions of feature space are structurally similar.

In [ ]:
leaf1 = tree1.apply(X_test)
leaf2 = tree2.apply(X_test)

partition_similarity = adjusted_rand_score(leaf1, leaf2)

print("Structural Stability")
print("--------------------")
print("Adjusted Rand Index between leaf partitions:", partition_similarity)
print()
print("Interpretation:")
print("ARI close to 1 means the two trees group test observations similarly.")
print("ARI close to 0 means the two trees produce very different partitions.")

## Compare Top-Level Splits

In [ ]:
feature_names = X_ames.columns

def get_top_splits(tree, feature_names, max_depth=3):
    tree_ = tree.tree_
    rows = []

    def recurse(node, depth):
        if depth > max_depth:
            return

        feature_index = tree_.feature[node]
        threshold = tree_.threshold[node]

        if feature_index != -2:
            rows.append({
                "depth": depth,
                "node": node,
                "feature": feature_names[feature_index],
                "threshold": threshold
            })

            recurse(tree_.children_left[node], depth + 1)
            recurse(tree_.children_right[node], depth + 1)

    recurse(0, 0)
    return pd.DataFrame(rows)

top_tree1 = get_top_splits(tree1, feature_names)
top_tree2 = get_top_splits(tree2, feature_names)

print("Top splits of Tree 1")
display(top_tree1)

print("Top splits of Tree 2")
display(top_tree2)

common_features = set(top_tree1["feature"]).intersection(set(top_tree2["feature"]))

print("Common top-level features:", common_features)
print("Number of common top-level features:", len(common_features))

## Prediction Comparison Plot

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(pred1, pred2, alpha=0.35)

min_val = min(pred1.min(), pred2.min())
max_val = max(pred1.max(), pred2.max())

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    color="red",
    linestyle="--",
    label="Perfect agreement"
)

plt.xlabel("Tree 1 predictions")
plt.ylabel("Tree 2 predictions")
plt.title("Ames Housing: Prediction Comparison Between Two Trees")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Visualize the First Three Levels

In [ ]:
plt.figure(figsize=(22, 10))
plot_tree(
    tree1,
    feature_names=feature_names,
    filled=True,
    max_depth=3,
    fontsize=7
)
plt.title("Tree 1: First Three Levels")
plt.show()

plt.figure(figsize=(22, 10))
plot_tree(
    tree2,
    feature_names=feature_names,
    filled=True,
    max_depth=3,
    fontsize=7
)
plt.title("Tree 2: First Three Levels")
plt.show()

## Repeat the Experiment

In [ ]:
results = []

for seed in range(10):
    X_sub = X_train.sample(frac=0.90, random_state=seed)
    y_sub = y_train.loc[X_sub.index]

    temp_tree = DecisionTreeRegressor(random_state=1)
    temp_tree.fit(X_sub, y_sub)

    temp_pred = temp_tree.predict(X_test)
    temp_leaf = temp_tree.apply(X_test)

    results.append({
        "seed": seed,
        "prediction_difference": np.mean(np.abs(pred1 - temp_pred)),
        "partition_similarity_ARI": adjusted_rand_score(leaf1, temp_leaf),
        "depth": temp_tree.get_depth(),
        "nodes": temp_tree.tree_.node_count
    })

results_df = pd.DataFrame(results)

display(results_df)

print("Average prediction difference:", results_df["prediction_difference"].mean())
print("Average partition similarity ARI:", results_df["partition_similarity_ARI"].mean())
print("Average depth:", results_df["depth"].mean())
print("Average number of nodes:", results_df["nodes"].mean())

## BQ1 Write-Up Text

You can adapt this paragraph for your final PDF:

I measured structural instability by comparing the leaf partitions produced by two trees trained on slightly different samples of the Ames Housing dataset. Each test observation was passed through both trees, and the leaf node assigned by each tree was recorded. If two trees are structurally similar, then observations grouped together in one tree should also be grouped together in the other tree. I used the Adjusted Rand Index to compare these leaf assignments. I also compared the top-level split features, tree depth, number of nodes, and the average absolute difference between predictions. The results show that small changes in training data can change the internal structure of the tree, including its depth, number of nodes, and some split rules. However, the prediction scatter plot shows that the two trees can still produce broadly similar predictions. This suggests that decision trees are structurally unstable even when their overall predictive behavior remains moderately similar.

# BQ2: Trees as Rule-Based Systems

A trained decision tree can be converted into explicit if-then rules. In this section, a shallow tree is trained on the Breast Cancer Wisconsin dataset, its rules are printed, and the same rules are implemented manually.

## Load Breast Cancer Wisconsin Dataset

In [ ]:
cancer = load_breast_cancer(as_frame=True)

X_bc = cancer.data
y_bc = cancer.target

X_train_bc, X_test_bc, y_train_bc, y_test_bc = train_test_split(
    X_bc,
    y_bc,
    test_size=0.25,
    random_state=42,
    stratify=y_bc
)

clf = DecisionTreeClassifier(max_depth=3, random_state=42)
clf.fit(X_train_bc, y_train_bc)

tree_predictions = clf.predict(X_test_bc)

print("BQ2 Results")
print("-----------")
print("Decision tree accuracy:", accuracy_score(y_test_bc, tree_predictions))
print("Classes:", cancer.target_names)

## Extract Rules

In [ ]:
rules = export_text(
    clf,
    feature_names=list(X_bc.columns)
)

print(rules)

## Visualize the Tree

In [ ]:
plt.figure(figsize=(20, 9))
plot_tree(
    clf,
    feature_names=X_bc.columns,
    class_names=cancer.target_names,
    filled=True,
    rounded=True,
    fontsize=8
)
plt.title("Breast Cancer Decision Tree as If-Then Rules")
plt.show()

## Print the Tree as Python-Style Rules

In [ ]:
def tree_to_code(tree, feature_names):
    tree_ = tree.tree_
    feature_name = [
        feature_names[i] if i != -2 else "leaf"
        for i in tree_.feature
    ]

    def recurse(node, depth):
        indent = "    " * depth

        if tree_.feature[node] != -2:
            name = feature_name[node]
            threshold = tree_.threshold[node]

            print(f"{indent}if row['{name}'] <= {threshold:.6f}:")
            recurse(tree_.children_left[node], depth + 1)

            print(f"{indent}else:")
            recurse(tree_.children_right[node], depth + 1)

        else:
            class_id = np.argmax(tree_.value[node][0])
            class_name = cancer.target_names[class_id]
            print(f"{indent}return {class_id}  # {class_name}")

tree_to_code(clf, list(X_bc.columns))

## Manual Rule-Based Prediction

The function below implements the rules produced by the tree. If your printed rules differ because of a different package version, copy the rules from the previous cell into this function.

In [ ]:
def manual_tree_predict(row):
    if row["worst radius"] <= 16.80:
        if row["worst concave points"] <= 0.14:
            if row["area error"] <= 91.56:
                return 1  # benign
            else:
                return 0  # malignant
        else:
            if row["worst texture"] <= 25.62:
                return 1  # benign
            else:
                return 0  # malignant
    else:
        if row["texture error"] <= 0.47:
            return 1  # benign
        else:
            if row["worst concavity"] <= 0.19:
                return 1  # benign
            else:
                return 0  # malignant

manual_predictions = X_test_bc.apply(manual_tree_predict, axis=1).values

comparison = pd.DataFrame({
    "actual_class": y_test_bc.values,
    "tree_prediction": tree_predictions,
    "manual_rule_prediction": manual_predictions
})

display(comparison.head(20))

print("Manual predictions match tree predictions:", np.all(tree_predictions == manual_predictions))
print("Number of mismatches:", np.sum(tree_predictions != manual_predictions))
print("Manual rule accuracy:", accuracy_score(y_test_bc, manual_predictions))

## Check Mismatches

In [ ]:
mismatches = comparison[
    comparison["tree_prediction"] != comparison["manual_rule_prediction"]
]

if len(mismatches) == 0:
    print("No mismatches: manual rules reproduce the trained tree exactly.")
else:
    display(mismatches)
    print("There are mismatches. Update the manual if-else function using the printed rules.")

## BQ2 Write-Up Text

You can adapt this paragraph for your final PDF:

I trained a shallow Decision Tree Classifier on the Breast Cancer Wisconsin dataset and extracted the learned decision rules. Each root-to-leaf path forms one if-then rule. I then implemented these rules manually using ordinary if-else statements and applied them to the test set. The manual rule-based predictions matched the trained decision tree predictions, showing that a fitted decision tree can be used outside a standard machine-learning pipeline as an explicit rule-based system. This is useful when interpretability, transparency, or lightweight deployment is important. However, the approach becomes harder to manage when the tree is deep, because the number of rules increases quickly. The extracted rules are also unstable: if the training data changes, the rules may change as well.